In [ ]:
import torch
import pickle
import matplotlib.pyplot as plt
import numpy as np
import csv
import math

# -----------------------------
# Load the data from the pickle file.
# The data is assumed to be a list of vehicles, and for each vehicle i,
# vehicle[i] is a torch tensor of shape [N, 6] with columns:
# [time, x, y, bbox_width, bbox_length, bbox_height]
with open('../i24_data/smoothed_all.pkl', 'rb') as f:
    smoothed = pickle.load(f)

# -----------------------------
# Define configuration parameters.
class Config:
    frame_rate_orignial = 25  # example frame rate
    frame_rate = 5
    dt = 1./frame_rate
    kalman_filter = 'constant acceleration'

cfg = Config()

# Define lane marking edges in meters.
# Note: These markings denote the edges, not the centers.
lane_edges = - torch.tensor([1, 2, 3, 4, 5.25], dtype=torch.float32) * 12 * 0.3048
# Convert to a sorted Python list (they are already in increasing order).
lane_edges_list = lane_edges.tolist()

def assign_lane_by_edges(y, edges):
    """
    Given a lateral y position (in meters) and a sorted list of lane edge positions,
    assign a lane id as follows:
      - If y is less than the first edge -> lane 1.
      - If y is between edges i-1 and i -> lane i+1.
      - If y is greater than or equal to the last edge -> lane len(edges)+1.
    """
    for idx, edge in enumerate(edges):
        if y > edge:
            return idx + 1
    return len(edges) + 1

# Open (or create) the CSV file for writing.
with open('../i24_data/output.csv', mode='w', newline='') as csv_file:
    writer = csv.writer(csv_file)
    
    # Write the header row.
    writer.writerow(['ID', 'time', 'x', 'y', 'l', 'w', 'h', 'lane'])
    
    # Loop over each vehicle
    for vehicle_id, vehicle in enumerate(smoothed):
        # Get the torch tensor for this vehicle.
        tensor_data = vehicle[2]
        
        # Ensure tensor is on CPU (if it's on GPU) and then convert it to a NumPy array.
        if tensor_data.is_cuda:
            tensor_data = tensor_data.cpu()
        data_np = tensor_data.numpy()
        
        # 1. Extract the time column.
        time_values = data_np[:, 0]
        
        # 2. Create a master time grid that is snapped to exact multiples of 0.2 seconds.
        # Compute the starting grid time as the smallest multiple of 0.2 that is >= the minimum time.
        start_time = math.ceil(time_values.min() / 0.2) * 0.2
        # Optionally, you can compute the end time as well; here we simply use max time plus a step to ensure coverage.
        master_time = np.arange(start_time, time_values.max() + 0.2, 0.2)
        
        # 3. Interpolate all other columns onto master_time.
        # Data columns: [x, y, bbox_width, bbox_length, bbox_height] (columns 1 to 5).
        # First, build a list with the master time grid.
        interp_columns = [master_time]
        for col in range(1, data_np.shape[1]):
            col_data = data_np[:, col]
            # Perform linear interpolation onto the master_time grid.
            interp_values = np.interp(master_time, time_values, col_data)
            interp_columns.append(interp_values)
        
        # 4. Combine the interpolated columns into one array.
        # The interpolated data now has shape [len(master_time), 6] and the columns are in the order:
        # [time, x, y, bbox_width, bbox_length, bbox_height]
        interp_data = np.column_stack(interp_columns)
        
        # 5. Write out each row to CSV.
        # Reorder each row into [ID, time, x, y, l, w, h]
        # where "l" is bbox_length (column index 4) and "w" is bbox_width (column index 3).
        for row in interp_data:
            time_val, x_val, y_val, bbox_w, bbox_l, bbox_h = row
            # Convert positions and bbox dimensions:
            w_meter = bbox_w * 0.3048
            l_meter = bbox_l * 0.3048
            h_meter = bbox_h * 0.3048
            x_meter = - x_val * 0.3048 - l_meter # TODO: CHECK THIS LATER!
            y_meter = y_val * 0.3048
            
            # Assign a lane id based on y position.
            # Assign lane id based on lateral position y_meter using lane edge markings.
            lane_id_assigned = assign_lane_by_edges(y_meter, lane_edges_list)
            
            # Write the row with the order: [ID, time, x, y, l, w, h, lane]
            writer.writerow([vehicle_id, time_val, x_meter, y_meter, l_meter, w_meter, h_meter, lane_id_assigned])

print("CSV file 'output.csv' created with interpolated data.")

In [ ]:
import torch
import pickle
import matplotlib.pyplot as plt
import numpy as np
import csv
import math
import pandas as pd

# Load the interpolated vehicle data from the CSV file.
df = pd.read_csv('../i24_data/output.csv')

In [ ]:
# Set up a figure for plotting.
plt.figure(figsize=(10, 8))

# Loop over each unique vehicle ID.
for vehicle_id in df['ID'].unique()[6000:10000]:
    # Extract data for the current vehicle.
    vehicle_df = df[df['ID'] == vehicle_id]
    
    # Plot the trajectory using x vs. y.
    plt.plot(vehicle_df['x'], vehicle_df['y'], label=f'Vehicle {vehicle_id}')
    
# Plot lanes
y_lanes = - torch.tensor([1, 2, 3, 4, 5.25]) * 12 * 0.3048
# Define an x-axis range (e.g., from 0 to 100 meters).
x = np.linspace(-1000, -3500, 500)
# Plot each lane as a horizontal line at its respective y-coordinate.
for y in y_lanes:
    plt.plot(x, [y.item()] * len(x), 'k', linewidth=2.5, label=f"Lane at y = {y.item():.2f} m")
    
# Label the axes and provide a title.
plt.xlabel('x-coordinate')
plt.ylabel('y-coordinate')
plt.title('Vehicle Trajectories (x-y)')

# Optionally, add a legend (if the number of vehicles is not too large).
# plt.legend(loc='best', fontsize='small')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Define a mapping from lane id to a color.
lane_colors = {
    1: 'blue',
    2: 'green',
    3: 'red',
    4: 'orange',
    5: 'purple'
}

plt.figure(figsize=(10, 8))

# Loop over each unique vehicle ID (e.g., the first 1000 vehicles).
for vehicle_id in df['ID'].unique()[6000:10000]:
    # Extract and sort data by time for the current vehicle.
    vehicle_df = df[df['ID'] == vehicle_id].sort_values('time')
    
    # Create grouping indices such that each group has a constant lane id.
    # This detects a change in lane by comparing each row with the previous one.
    group_ids = (vehicle_df['lane'] != vehicle_df['lane'].shift()).cumsum()
    
    # Group the vehicle's data into segments where lane id stays constant.
    for _, group in vehicle_df.groupby(group_ids):
        lane_id = group['lane'].iloc[0]  # lane id for this segment
        color = lane_colors.get(lane_id, 'black')
        # Plot the segment using the corresponding lane color.
        plt.plot(group['x'], group['y'], color=color)
        
# Plot lane markings.
# Compute lane center positions in meters; note the minus sign if your y's are negative.
y_lanes = -torch.tensor([1, 2, 3, 4, 5.25], dtype=torch.float32) * 12 * 0.3048
# Define an x-axis range.
x = np.linspace(-1000, -3500, 500)
for idx, y in enumerate(y_lanes):
    plt.plot(x, [y.item()] * len(x), 'k', linewidth=2.5, label=f"Lane {idx+1} at y = {y.item():.2f} m")

plt.xlabel('x-coordinate')
plt.ylabel('y-coordinate')
plt.title('Vehicle Trajectories (x-y) with Lane Color Coding (time-varying lanes)')
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Load the CSV file that contains columns: 
# [ID, time, x, y, l, w, h, lane]
# df = pd.read_csv("output.csv")

# Create an empty column to store the follower ID.
df["follower_id"] = np.nan

# Assuming that vehicles are on a common time stamp grid, we can group by "time" and "lane".
# (If the time stamps differ across vehicles, you may need to use a tolerance or merge on nearest time.)
grouped = df.groupby(["time", "lane"])

# Process each group (i.e. each combination of time stamp and lane).
for (t, lane), group in grouped:
    # Sort by x coordinate in descending order.
    # The leader (with the highest x) will be at the top of this sorted group.
    group_sorted = group.sort_values(by="x", ascending=False)
    
    # For every car except the last, assign the follower id to be the id of the car immediately behind.
    follower_ids = [None] * len(group_sorted)
    
    # Iterate over the sorted group: the car at index i is followed by the car at i+1.
    for i in range(len(group_sorted) - 1):
        follower_ids[i] = group_sorted.iloc[i + 1]["ID"]
        
    # The last car in the sorted order has no follower.
    follower_ids[-1] = np.nan
    
    # Assign these follower IDs back to the original DataFrame.
    df.loc[group_sorted.index, "follower_id"] = follower_ids

# Optionally, sort the DataFrame by time and lane again.
df.sort_values(by=["time", "lane", "x"], ascending=[True, True, False], inplace=True)

# Save the updated DataFrame to a new CSV file.
df.to_csv("../i24_data/output_with_followers.csv", index=False)

print("Processed CSV file created: 'output_with_followers.csv'.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the CSV file.
# df = pd.read_csv("output_with_followers.csv")

# Choose a specific leader vehicle ID.
target_vehicle_id = 56  # change this to your desired vehicle ID

# Extract the leader's trajectory, sorted by time.
leader_df = df[df["ID"] == target_vehicle_id].sort_values("time")

# Set up the plot.
plt.figure(figsize=(10, 8))
plt.plot(leader_df["time"], leader_df["x"], 'k-', linewidth=3, label=f"Leader {target_vehicle_id}")

# Find the unique follower IDs over the leader's trajectory.
# (Remove NaNs if any exist.)
unique_followers = leader_df["follower_id"].dropna().unique()

# Create a colormap for different follower trajectories.
# Use a qualitative colormap with as many colors as there are unique followers.
colors = plt.cm.get_cmap("tab10", len(unique_followers))

# For each follower, extract and plot its trajectory over the corresponding time stamps.
for idx, f_id in enumerate(unique_followers):
    f_id = int(f_id)  # convert to integer if needed
    # Option 1: extract time stamps from the leader when the follower is f_id,
    leader_times = leader_df.loc[leader_df["follower_id"] == f_id, "time"]
    # Extract the follower's trajectory where its ID equals f_id and the time is in leader_times.
    follower_df = df[(df["ID"] == f_id) & (df["time"].isin(leader_times))].sort_values("time")
    
    if follower_df.empty:
        print(f"No trajectory data found for follower {f_id} over the corresponding times.")
        continue

    # Plot the follower's trajectory with a dashed line and a unique color.
    plt.plot(follower_df["time"], follower_df["x"],
             linestyle='--', linewidth=2, color=colors(idx),
             label=f"Follower {f_id}")
    
# Label the plot.
plt.xlabel("time (sec)")
plt.ylabel("x (meters)")
plt.title(f"Trajectories for Leader {target_vehicle_id} and its Follower(s)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Load the CSV file containing vehicle trajectories.
# Expected columns (at least): 
#   "ID"         : vehicle identifier (each row corresponds to that vehicle’s state at a time)
#   "time"       : time stamp (in seconds) -- assumed common across vehicles, but may require rounding
#   "x", "y"     : positions (in meters; after coordinate transformation)
#   "lane"       : lane id
#   "l", "w", "h": vehicle dimensions (in meters)
#   "follower_id": the id of the vehicle that is following the leader at that time stamp
# -----------------------------

df = pd.read_csv("../i24_data/output_with_followers.csv")

# To help with matching floating point time values, round the time column.
df["time"] = df["time"].round(1)

# # Remove rows with missing follower information.
# df = df.dropna(subset=["follower_id"])

# # Ensure proper data types.
# df["follower_id"] = df["follower_id"].astype(int)

# Sort by vehicle ID and time.
df = df.sort_values(["ID", "time"]).reset_index(drop=True)

# -----------------------------
# Define the minimum duration threshold for a car-following pair (in seconds).
threshold = 30.0  # adjust as needed

# List for storing extracted pairs.
pairs = []

# Group by leader vehicle (here the "ID" indicates the vehicle acting as leader in its own rows).
for leader_id, leader_group in df.groupby("ID"):
    # Sort the leader's data by time.
    leader_group = leader_group.sort_values("time").reset_index(drop=True)
    
    # Create contiguous segments when the follower remains constant.
    # A change in follower_id signals a new segment.
    leader_group["seg_id"] = (leader_group["follower_id"].ne(leader_group["follower_id"].shift())).cumsum()
    
    for seg, seg_df in leader_group.groupby("seg_id"):
        # Compute the duration of the segment.
        start_time = seg_df["time"].iloc[0]
        end_time = seg_df["time"].iloc[-1]
        duration = end_time - start_time
        
        # Skip segments shorter than the threshold.
        if duration < threshold:
            continue
        
        # Get the follower id for this segment (should be constant within the segment).
        follower_id = seg_df["follower_id"].iloc[0]
        
        # The common time stamps for this segment.
        common_times = seg_df["time"].values   # already rounded
        
        # --- Now, extract the corresponding follower trajectory.
        # Get all rows for the follower.
        follower_all = df[df["ID"] == follower_id].copy()
        # Round follower times too.
        follower_all["time"] = follower_all["time"].round(2)
        
        # Select follower rows that occur during the leader's common times.
        follower_seg = follower_all[follower_all["time"].isin(common_times)]
        # In case of any ordering issues, sort by time.
        follower_seg = follower_seg.sort_values("time").reset_index(drop=True)
        
        if follower_seg.empty:
            continue
        
        # Compute the common intersection of times (this handles any minor mismatches).
        common_times_follower = np.intersect1d(common_times, follower_seg["time"].values)
        if len(common_times_follower) == 0:
            continue
        
        # Restrict leader segment to those common times.
        leader_seg_common = seg_df[seg_df["time"].isin(common_times_follower)]
        leader_x = leader_seg_common["x"].values
        leader_y = leader_seg_common["y"].values
        leader_tidx_common = leader_seg_common.index.values  # indices within this segment
        times_common = leader_seg_common["time"].values
        
        # Similarly, restrict the follower data to the common times.
        follower_seg_common = follower_seg[follower_seg["time"].isin(common_times_follower)]
        follower_seg_common = follower_seg_common.sort_values("time").reset_index(drop=True)
        follower_x = follower_seg_common["x"].values
        follower_y = follower_seg_common["y"].values
        follower_tidx = follower_seg_common.index.values
        follower_w = follower_seg_common["w"].values
        
        # --- Compute additional quantities.
        # Gap: compute as the difference between leader (rear) and follower (front) positions.
        # (Assuming x is the longitudinal coordinate and leader_x > follower_x when the leader is ahead.)
        gap = leader_x - follower_x - follower_w
        
        # Compute speeds via numerical differentiation using np.gradient.
        # Leader speed (in m/s):
        leader_speed = np.gradient(leader_x, times_common)
        leader_speed[-1] = leader_speed[-2]
        # Follower speed (in m/s):
        follower_speed = np.gradient(follower_x, times_common)
        follower_speed[-1] = follower_speed[-2]
        
        # Relative speed:
        relative_speed = follower_speed - leader_speed
        
        # Build the pair dictionary.
        pair = {
            "leader_id": leader_id,
            "leader_tidx": leader_tidx_common,   # indices of leader rows for this segment
            "follower_id": follower_id,
            "follower_tidx": follower_tidx,       # indices of follower rows in its own trajectory
            "time": times_common,                 # common time stamps for the segment
            "leader_x": leader_x,                 # leader x positions (m)
            "leader_y": leader_y,                 # leader y positions (m)
            "follower_x": follower_x,             # follower x positions (m)
            "follower_y": follower_y,             # follower y positions (m)
            "gap": gap,                         # gap between leader and follower (m)
            "leader_speed": leader_speed,         # computed leader speed (m/s)
            "follower_speed": follower_speed,     # computed follower speed (m/s)
            "relative_speed": relative_speed,      # leader_speed - follower_speed (m/s)
            "vehicle_length": follower_w
        }
        
        pairs.append(pair)

# Report the number of extracted pairs.
print("Extracted", len(pairs), "car-following pairs longer than the threshold.")

# For demonstration, print out the first pair's keys and the shapes of its arrays.
if pairs:
    first_pair = pairs[0]
    print("Example pair keys and data shapes:")
    for key, value in first_pair.items():
        if isinstance(value, np.ndarray):
            print(f" {key}: {value.shape}")
        else:
            print(f" {key}: {value}")


# Optionally, you can save these pairs (e.g., using pickle) for further analysis.
import pickle
with open("../i24_data/car_following_pairs.pkl", "wb") as f:
    pickle.dump(pairs, f)

print("Car-following pairs have been saved to 'car_following_pairs.pkl'.")


In [ ]:
import pickle
import numpy as np

# Load existing pairs
with open("../i24_data/car_following_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)

# Parameters
MAX_GAP = 200.0      # maximum allowable gap (m) for a car-following event
MIN_DURATION = 20.0  # optional: minimum duration in seconds

filtered_pairs = []
reject_log = []

for pair in pairs:
    gap = np.array(pair["gap"])           # (T,)
    time_vals = np.array(pair["time"])    # (T,)

    # Basic checks
    always_positive = np.all(gap > 0)
    max_gap = gap.max() if gap.size else np.inf
    duration = time_vals[-1] - time_vals[0] if time_vals.size else 0

    # Classify
    if always_positive and max_gap <= MAX_GAP and duration >= MIN_DURATION:
        filtered_pairs.append(pair)
    else:
        reject_log.append({
            "leader_id": pair["leader_id"],
            "follower_id": pair["follower_id"],
            "max_gap": float(max_gap),
            "mean_gap": float(gap.mean()) if gap.size else np.nan,
            "duration": float(duration),
            "reason": (
                [] 
                + (["gap not positive"] if not always_positive else [])
                + (["gap too large"] if max_gap > MAX_GAP else [])
                + (["too short"] if duration < MIN_DURATION else [])
            )
        })

print(f"Kept {len(filtered_pairs)} pairs, removed {len(reject_log)} pairs.\n")

# Optional: inspect which pairs were removed and why
for r in reject_log:
    print(f"Removed Leader {r['leader_id']} – Follower {r['follower_id']}: "
          f"max_gap={r['max_gap']:.1f} m, mean_gap={r['mean_gap']:.1f} m, "
          f"duration={r['duration']:.1f}s → {', '.join(r['reason'])}")


In [ ]:
import matplotlib.pyplot as plt

# Report the number of remaining pairs.
print("We keep", len(filtered_pairs), "car-following pairs longer than the threshold.")
discard = 10

# For demonstration, we plot all the pairs.
for idx, pair in enumerate(filtered_pairs[:10]):
    # Extract common data from the pair dictionary.
    time_vals = np.array(pair["time"])[:-discard]  # common time stamps (s)
    leader_x = np.array(pair["leader_x"])[:-discard]  # leader x positions (m)
    leader_y = np.array(pair["leader_y"])[:-discard]  # leader y positions (m)
    follower_x = np.array(pair["follower_x"])[:-discard]  # follower x positions (m)
    follower_y = np.array(pair["follower_y"])[:-discard]  # follower y positions (m)
    gap = np.array(pair["gap"])[:-discard]  # gap between leader and follower (m)
    leader_speed = np.array(pair["leader_speed"])[:-discard]  # leader speed (m/s)
    follower_speed = np.array(pair["follower_speed"])[:-discard]  # follower speed (m/s)
    relative_speed = np.array(pair["relative_speed"])[:-discard]  # relative speed (m/s)

    # Create a figure with multiple subplots.
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f"Car-following Pair {idx+1}: Leader {pair['leader_id']} and Follower {pair['follower_id']}",
                 fontsize=14)

    # ---------------------------
    # Subplot 1: Spatial Trajectories
    axs[0, 0].plot(time_vals, leader_x, 'b--.', linewidth=2, label=f"Leader {pair['leader_id']}")
    axs[0, 0].plot(time_vals, follower_x, 'r--.', linewidth=2, label=f"Follower {pair['follower_id']}")
    axs[0, 0].set_xlabel("time (sec)")
    axs[0, 0].set_ylabel("x (m)")
    axs[0, 0].set_title("Spatial Trajectories")
    axs[0, 0].legend()
    axs[0, 0].grid(True)

    # ---------------------------
    # Subplot 2: Gap vs. Time
    axs[0, 1].plot(time_vals, gap, 'g-.', linewidth=2)
    axs[0, 1].set_xlabel("Time (s)")
    axs[0, 1].set_ylabel("Gap (m)")
    axs[0, 1].set_title("Gap vs. Time")
    axs[0, 1].grid(True)

    # ---------------------------
    # Subplot 3: Speeds vs. Time
    axs[1, 0].plot(time_vals, leader_speed, 'b-', linewidth=2, label="Leader Speed")
    axs[1, 0].plot(time_vals, follower_speed, 'r--', linewidth=2, label="Follower Speed")
    axs[1, 0].set_xlabel("Time (s)")
    axs[1, 0].set_ylabel("Speed (m/s)")
    axs[1, 0].set_title("Speeds vs. Time")
    axs[1, 0].legend()
    axs[1, 0].grid(True)

    # ---------------------------
    # Subplot 4: Relative Speed vs. Time
    axs[1, 1].plot(time_vals, relative_speed, 'k-', linewidth=2)
    axs[1, 1].set_xlabel("Time (s)")
    axs[1, 1].set_ylabel("Relative Speed (m/s)")
    axs[1, 1].set_title("Relative Speed vs. Time")
    axs[1, 1].grid(True)

    # Adjust layout and show the figure.
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
# Optionally, you can save these pairs (e.g., using pickle) for further analysis.
import pickle
with open("../i24_data/filtered_car_following_pairs.pkl", "wb") as f:
    pickle.dump(filtered_pairs, f)

In [ ]:
import pickle
fp = open("../i24_data/filtered_car_following_pairs.pkl", 'rb')
cf_pair_dicts = pickle.load(fp)

# stack the data into collections
count = 0
for cf_pair in cf_pair_dicts:
    if count == 0:
        collection_s = np.array(cf_pair['gap'])
        collection_v = np.array(cf_pair['follower_speed'])
        collection_a = np.array(cf_pair['leader_speed'])
        collection_dv = np.array(cf_pair['relative_speed'])
    else:
        collection_s = np.hstack((collection_s, cf_pair['gap']))
        collection_v = np.hstack((collection_v, cf_pair['follower_speed']))
        collection_a = np.hstack((collection_a, cf_pair['leader_speed']))
        collection_dv = np.hstack((collection_dv, cf_pair['relative_speed']))

    count += 1

# Plot the histogram of the data
plt.figure(figsize=(10, 6))
plt.subplot(2, 2, 1)
plt.hist(collection_s, bins=50, alpha=0.5, label='gap', color='blue')
plt.legend()
plt.grid()
plt.subplot(2, 2, 2)
plt.hist(collection_v, bins=50, alpha=0.5, label='follower speed', color='red')
plt.grid()
plt.legend()
plt.subplot(2, 2, 3)
plt.hist(collection_a, bins=50, alpha=0.5, label='leader speed', color='green')
plt.grid()
plt.legend()
plt.subplot(2, 2, 4)
plt.hist(collection_dv, bins=50, alpha=0.5, label='relative speed', color='orange')
plt.grid()
plt.legend()
plt.show()